# Decorators Without the Magic-Trick Feel
This notebook has been rewritten to be slower, more reflective, and less template-like. Instead of treating **Decorators Without the Magic-Trick Feel** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


When people struggle with **Decorators Without the Magic-Trick Feel**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- See decorator syntax as ordinary function transformation
- Understand why wrappers need *args and **kwargs
- Preserve metadata with wraps
- Read decorated code with less fear


At runtime, the original function object is still an object. The decorator receives it, builds a wrapper object that closes over it, and then the name is rebound to that wrapper. The memory-level story is just “new function object, new binding”.


In [1]:
from functools import wraps

def logger(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print("calling", fn.__name__)
        return fn(*args, **kwargs)
    return wrapper

@logger
def greet(name):
    return f"Hello, {name}"

print(greet("Ada"))
print(greet.__name__)


calling greet
Hello, Ada
greet


Disassembly is useful because it reminds us that wrapper functions are still just functions, and decorated names now point to those wrappers. The syntax is special, but the runtime object model is not.


In [2]:
import dis

def plain(name):
    return f"Hello, {name}"

dis.dis(plain)


  3           0 RESUME                   0

  4           2 LOAD_CONST               1 ('Hello, ')
              4 LOAD_FAST                0 (name)
              6 FORMAT_VALUE             0
              8 BUILD_STRING             2
             10 RETURN_VALUE


That is the real foundation.

That is why it can call the original later.

Without it, debugging decorated functions becomes more annoying.

They let you configure the wrapping behavior.


Seeing the manual transformation once is worth a lot.


In [3]:
def shout(fn):
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs).upper()
    return wrapper

def greet(name):
    return f"Hello, {name}"

greet = shout(greet)
print(greet("Ada"))


HELLO, ADA


Now the sugar feels earned instead of magical.


In [4]:
from functools import wraps

def timed(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        import time
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        print(time.perf_counter() - start)
        return result
    return wrapper

@timed
def work():
    return sum(range(100000))

print(work())


0.0016065000017988496
4999950000


The outer function accepts configuration. The inner function becomes the actual decorator.


In [5]:
from functools import wraps

def repeat(times):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = fn(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(2)
def wave():
    print("wave")

wave()


wave
wave


This is not only a classroom topic. **Decorators Without the Magic-Trick Feel** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Learning decorators only from `@` syntax
- Forgetting `functools.wraps`
- Writing wrappers that do not forward all arguments correctly


- What is a decorator really?
- Why do wrappers often take *args and **kwargs?
- Why use `functools.wraps`?


- Decorators wrap functions.
- The `@` form is syntax sugar.
- Closures and first-class functions make decorators possible.


If this notebook did its job, **Decorators Without the Magic-Trick Feel** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.
